# Moroccan Darija ASR: 3h of data, one adapted model

**Task.** Adapt a strong Arabic ASR model to Moroccan Darija with a small budget: 3 hours of audio, one 16GB GPU, a few hours of training.

**Data.** I collected Darija YouTube audio, cleaned it with a VAD + quality pipeline, and labeled it with Gemini 2.5 Pro. The result is published as [`01Yassine/darija-asr-3h`](https://huggingface.co/datasets/01Yassine/darija-asr-3h): 3.00h train, 0.15h validation, 0.35h silver, split by channel.

**Model.** I benchmarked stock checkpoints on [`atlasia/darija-asr-benchmark`](https://huggingface.co/datasets/atlasia/darija-asr-benchmark) (114 clips, human labels) and picked `CohereLabs/cohere-transcribe-arabic-07-2026`: 19.9 CER zero-shot.

**Method.** Freeze the 2B Conformer encoder. Train a small conv adapter inside the encoder plus LoRA on the decoder: 21.1M of 2.09B parameters, or 1.0%.

**Where things stand.** All four recipes finished. On validation (Gemini, 91 clips) **hybrid** (MultiConv + LoRA) and `full_lora` both land at 14.9 CER. On AtlasIA (human, 114 clips) the tie breaks: **hybrid is 14.4 CER**, from a 20.2 zero-shot on the same eval script. Encoder-only and decoder-only barely move.

| section | what is in it |
| --- | --- |
| 1 | setup and every knob in one cell |
| 2 | how the 3h set was built, and the set itself |
| 3 | scoring rules (CER, WER, bootstrap) |
| 4 | zero-shot table, and how to rerun it |
| 5 | the method: decisions, then all the training code |
| 6 | evaluation on validation, silver and AtlasIA |
| 7 | results and what I would do next |

Everything runs from this notebook and the Hub. No local dataset files are needed.


## Use the model

All four recipes are public Hugging Face adapters. Inference downloads the adapter from the Hub and loads Cohere Arabic under it. No local `checkpoints/` needed.

| recipe | Hub | AtlasIA CER |
| --- | --- | ---: |
| **hybrid** (MultiConv + LoRA) | [`01Yassine/cohere-transcribe-darija`](https://huggingface.co/01Yassine/cohere-transcribe-darija) | **14.4** |
| full LoRA | [`01Yassine/cohere-transcribe-darija-full-lora`](https://huggingface.co/01Yassine/cohere-transcribe-darija-full-lora) | 16.5 |
| encoder LoRA | [`01Yassine/cohere-transcribe-darija-encoder-lora`](https://huggingface.co/01Yassine/cohere-transcribe-darija-encoder-lora) | 17.4 |
| decoder LoRA | [`01Yassine/cohere-transcribe-darija-decoder-lora`](https://huggingface.co/01Yassine/cohere-transcribe-darija-decoder-lora) | 20.2 |

From the Hub only (no clone of this repo):

```python
from huggingface_hub import snapshot_download
import sys
sys.path.insert(0, snapshot_download("01Yassine/cohere-transcribe-darija"))
from infer import transcribe
print(transcribe("clip.wav"))
print(transcribe("clip.wav", model_id="full_lora"))
```

From this repo:

```bash
pip install "transformers>=5.4" peft torch torchaudio soundfile huggingface_hub
python infer.py clip.wav                      # hybrid
python infer.py clip.wav --model full_lora
python infer.py clip.wav --model encoder_lora
python infer.py clip.wav --model decoder_lora
```


## 1. Setup

Cohere ASR needs `transformers>=5.4`. A GPU is needed for section 4 onwards; sections 2 and 3 run on CPU.

This notebook is standalone: every class and function it needs is defined in it, so it runs from the Hub with no local files. The repo holds the same logic as scripts, which is what I actually launch on the cluster. The mapping:

| notebook | repo file | how it is run |
| --- | --- | --- |
| 2.1 to 2.6 | `data/collect/*.py`, `data/process/*.py` | commands in section 2 |
| 2.7 splits | `src/prepare_data.py` | `bash scripts/run_prepare.sh` |
| 3 scoring | `src/normalize.py`, `src/metrics.py` | imported everywhere |
| 4 zero-shot | `benchmarks/zeroshot_casablanca/{lineup,run,write_report}.py` | `bash scripts/srun_zeroshot_atlasia.sh` |
| 5.2 adapter, 5.3 LoRA | `src/adapters.py`, `src/cohere_runtime.py` | imported by training |
| 5.4 augmentation | `src/augment.py` | imported by training |
| 5.5 to 5.8 training | `src/train_cohere.py` | `bash scripts/srun_train_cohere.sh method`  (hybrid) |
| 6 evaluation | `scripts/eval_gold.py` | `bash scripts/srun_eval_atlasia.sh` |
| Hub inference | `infer.py` | `python infer.py clip.wav` |

Use the notebook to read and to run small things. Use the scripts for the real runs, since a 3 hour Slurm job should not depend on a live kernel.


In [ ]:
!pip install -q "transformers>=5.4" torch torchaudio peft accelerate datasets jiwer soundfile pandas numpy


In [ ]:
import json
import math
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
from datasets import Audio, load_dataset

# ---- everything you might want to change lives here ----
HF_DATASET = "01Yassine/darija-asr-3h"
GOLD_DATASET = "atlasia/darija-asr-benchmark"     # human eval set (split=train)
BASE_MODEL = "CohereLabs/cohere-transcribe-arabic-07-2026"
LANGUAGE = "ar"
SAMPLE_RATE = 16000
SEED = 42

# where checkpoints go. change if you cloned this elsewhere.
WORK_DIR = Path.cwd() / "runs"

RECIPE = "hybrid"          # hybrid | decoder_lora | full_lora | encoder_lora
EPOCHS = 5
TRAIN_BS = 8               # raise with the VRAM probe in 5.6
EVAL_BS = 1                # generate is the memory spike, keep it at 1
GRAD_ACCUM = 2
VRAM_BUDGET_GB = 16.0
MAX_STEPS = -1             # set e.g. 20 for a smoke test
MAX_NEW_TOKENS = 128
NUM_WORKERS = min(4, max(0, (os.cpu_count() or 2) - 1))

LORA = dict(r=32, lora_alpha=64, lora_dropout=0.05, bias="none",
            target_kinds=("q_proj", "k_proj", "v_proj", "o_proj"))
ADAPTER = dict(bottleneck=64, kernels=(7, 15, 23, 31), fusion="concat_fusion",
               merge_kernel=31, dropout=0.1, skip_bottom_frac=0.33)
LR_ADAPTER = 1e-4
LR_LORA = 1e-4

AUG = dict(speed_factors=(0.9, 1.0, 1.1), gain_db=3.0,
           drop_chunk_prob=0.3, drop_chunk_max_frac=0.15,
           noise_prob=0.25, noise_snr=(10.0, 20.0), rir_prob=0.25,
           specaug_n_time=2, specaug_n_freq=2,
           specaug_time_frac=0.10, specaug_freq_frac=0.10)
# optional noise / reverb corpora. leave None to skip them.
MUSAN_ROOT = None    # e.g. "/path/to/musan"
RIR_ROOT = None      # e.g. "/path/to/RIRS_NOISES"

RECIPES = {
    "hybrid":       dict(conv_adapter=True,  lora_scope="decoder"),  # MultiConv + decoder LoRA
    "decoder_lora": dict(conv_adapter=False, lora_scope="decoder"),
    "full_lora":    dict(conv_adapter=False, lora_scope="full"),
    "encoder_lora": dict(conv_adapter=False, lora_scope="encoder"),
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("device", DEVICE, "| recipe", RECIPE, RECIPES[RECIPE], "| work dir", WORK_DIR)


## 2. The data

3 hours is the budget, so the question is not how much audio I can get but which 3 hours are worth labeling. Every stage below exists to protect the labels: Gemini 2.5 Pro is a good Darija transcriber on clean audio, and a confident fabricator on bad audio.

Seven scripts, in this order. Each one adds columns to a single metadata parquet, so nothing gets recomputed:

```bash
python data/collect/crawl.py                                  # 2.1  audio from the channel list
python data/process/silero_vad.py                             # 2.2  cut into speech clips
python data/process/audio_aesthetics.py --input meta.parquet   # 2.3  SQUIM + Audiobox + loudness
python data/process/speech_enhancer.py meta_metadata.parquet \
       --pesq-threshold 1.75                                  # 2.4  denoise only the worst
PARQUET_PATH=... HF_TOKEN=... \
       python data/process/speaker_separator.py               # 2.5  num_speakers
python your_runner.py   # uses data/collect/transcriber.py    # 2.6  Gemini teacher labels
python -m src.prepare_data                                    # 2.7  filter, sample, split
```


### 2.1 Collect: `data/collect/crawl.py`

A list of 51 Moroccan Darija YouTube channels (vlogs, cooking, football talk, finance in Darija). Up to 70 uploads per channel, audio stream only. A channel list rather than keyword search gives a dialect prior and, more importantly, a channel id per clip, which is what makes the split in 2.7 possible.

```python
# data/collect/configs.py
CHANNEL_LIST_PATH = os.environ.get("CHANNEL_LIST_PATH", str(HERE / "channels.txt"))
NUMBER_SELECTED_VIDEOS = int(os.environ.get("NUMBER_SELECTED_VIDEOS", "70"))
FILE_NAME_PREFIX = os.environ.get("FILE_NAME_PREFIX", "darija")

# data/collect/crawl.py
def crawl(channels: list[str]) -> None:
    for handle in channels:
        channel_dir = os.path.join(SAVE_PATH, handle)
        os.makedirs(channel_dir, exist_ok=True)
        videos = Channel(URL_PREFIX + handle).videos
        for i, video in enumerate(videos[: min(NUMBER_SELECTED_VIDEOS, len(videos))]):
            try:
                video.streams.get_audio_only().download(      # audio track only, no video mux
                    output_path=channel_dir,
                    filename=f"@{FILE_NAME_PREFIX}_{i}_audio",
                )
            except Exception:
                continue        # private, region-locked or dead uploads
```

On disk: `{SAVE_PATH}/{@handle}/@darija_{i}_audio`. The handle in the path is the channel id used later.


### 2.2 Segment: `data/process/silero_vad.py`

A 40 minute video is not an utterance. Silero marks speech regions; each region becomes a clip, capped at 30s so one clip cannot swallow a video. Spans under 0.5s are dropped, since those are usually clicks, breaths or VAD mistakes. Music, intros and long silence disappear here.

```python
# data/process/config.py
MAX_SPEECH_DURATION_S = 30
MIN_SPEECH_DURATION_MS = 500     # below this it is a click or a breath, not speech
PESQ_ENHANCE_BELOW = 1.75
PESQ_TRAIN_ABOVE = 2.5

# data/process/silero_vad.py
model = load_silero_vad()
for wav_path in glob.glob(f"{base_dir}/*/wav/*.wav"):
    wav = read_audio(wav_path)
    timestamps = get_speech_timestamps(
        wav, model, return_seconds=True,
        max_speech_duration_s=MAX_SPEECH_DURATION_S,
        min_speech_duration_ms=MIN_SPEECH_DURATION_MS,
    )
    fname = Path(wav_path).stem
    out_dir = Path(wav_path).parent.parent / "silero_vad" / fname
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / f"{fname}.json").write_text(json.dumps(timestamps, indent=2))
    audio = AudioSegment.from_wav(wav_path)
    for i, ts in enumerate(timestamps):
        audio[int(ts["start"] * 1000):int(ts["end"] * 1000)].export(
            out_dir / f"{i}.wav", format="wav")
```

The timestamps json is kept next to the clips, so any clip can be traced back to its position in the source video.


### 2.3 Score quality: `data/process/audio_aesthetics.py`

No clean reference exists, so quality is estimated. SQUIM predicts what PESQ, STOI and SI-SDR *would* be if a reference existed, Audiobox rates production quality, and ffmpeg measures loudness.

```python
# data/process/audio_aesthetics.py
from torchaudio.pipelines import SQUIM_OBJECTIVE
from audiobox_aesthetics.infer import main_predict

@torch.inference_mode()
def compute_squim_scores(waveform, model, device):
    stoi_hyp, pesq_hyp, si_sdr_hyp = model(waveform[0])
    return float(pesq_hyp.item()), float(stoi_hyp.item()), float(si_sdr_hyp.item())

def get_ffmpeg_loudness(audio_path):        # near-silent or clipped means broken download
    command = ["ffmpeg", "-hide_banner", "-nostats", "-i", audio_path,
               "-filter_complex", "ebur128=dualmono=true:peak=true:framelog=quiet",
               "-f", "null", "-"]
    return parse_ffmpeg_output(subprocess.run(command, capture_output=True, text=True).stderr)

results = main_predict(metadata, ckpt=None, batch_size=batch_size)   # Audiobox
path_to_result = {valid_paths[i]: results_dicts[i] for i in range(len(valid_paths))}
df["content_enjoyment"] = df["path"].apply(lambda p: path_to_result.get(p, {}).get("CE", np.nan))
# same for CU -> content_usefulness, PC -> production_complexity, PQ -> production_quality
```

What each score is used for:

| score | measures | reading | used as |
| --- | --- | --- | --- |
| `pesq_hyp` | noise and distortion | 1 is very bad, 4.5 is clean | keep above 2.5 |
| `stoi_hyp` | intelligibility | 0 to 1, 0.9 means clear words | drop the low tail |
| `si_sdr_hyp` | speech vs distortion | dB, higher is cleaner | drop very negative |
| `PQ`, `CU`, `CE`, `PC` | recording quality, useful content | relative | drop low PQ / near-zero CU |
| LUFS, true peak | loudness and clipping | dBFS | drop near-silent or clipped |

Note that `pesq_hyp` is an estimate, not ITU P.862.


### 2.4 Denoise the rescuable: `data/process/speech_enhancer.py`

DNS64 runs only on clips with `pesq_hyp < 1.75`. Denoising everything smears clean recordings and Gemini copies the artefacts.

```python
# data/process/speech_enhancer.py
self.model = pretrained.dns64().to(self.device).eval()

subset = df[df["pesq_hyp"] < pesq_threshold].copy()      # threshold 1.75
for idx, row in subset.iterrows():
    waveform, orig_sr = torchaudio.load(row["path"])
    enhanced = denoiser.denoise_tensor(waveform, orig_sr)
    enhanced_path = make_enhanced_path(row["path"])       # silero_vad -> silero_vad_enhanced
    torchaudio.save(enhanced_path, enhanced, int(denoiser.sample_rate))
    new_row = row.copy()
    new_row["id"] = str(row["id"]) + "_Enh"               # added as a new row, original kept
    new_row["path"] = enhanced_path
    enhanced_rows.append(new_row)
```

The enhanced clip is a new row with an `_Enh` suffix, not a replacement, so both versions stay comparable. The train filter is still `pesq_hyp > 2.5`, so a clip that is still dirty after DNS64 is dropped rather than rescued.


### 2.5 Count speakers: `data/process/speaker_separator.py`

Gemini returns one transcript per file, so overlapping speech becomes mixed or missing turns. This is not source separation, just a refusal.

```python
# data/process/speaker_separator.py
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", use_auth_token=hf_token)
pipeline.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

for idx, row in df.iterrows():
    num_speakers = len(sorted(set(pipeline(row["path"]).labels())))
    df.at[idx, "num_speakers"] = num_speakers
```

`num_speakers` goes on the metadata and the 3h set keeps single-speaker clips.


### 2.6 Label: `data/collect/transcriber.py`

The prompt is the label contract:

```python
# data/collect/transcriber.py
DEFAULT_MODEL = "gemini-2.5-pro"
DEFAULT_PROMPT = """\
You are an audio transcriber. ...

Rules:
* Respond with an exact transcription of the audio input.
* Audio is Moroccan Darija Arabic.
* Do not transliterate: if words from another language are uttered (e.g. French,
  English, etc.), transcribe them in their original script (e.g. latin letters).
* Do not include any text other than the transcription.
* Return transcriptions in the order they were received, each in a newline.
* Format each line as: "<filename.wav>: transcription"
* If the audio is not clear, respond with "<filename.wav>:" (empty transcription).
"""
```

So Darija comes back in Arabic script, French and English stay in Latin script, and unclear clips come back empty and get dropped. About 30% of train clips contain code-switch.

Clips go through the Gemini batch API, 10 files per request, and the reply is parsed back per filename:

```python
# data/collect/transcriber.py
def _parse_response(self, text: str) -> dict[str, str]:
    results = {}
    for line in text.strip().splitlines():
        if ":" not in line:
            continue
        name, transcription = line.split(":", maxsplit=1)
        results[name.strip()] = transcription.strip()
    return results

transcriber = Transcriber(genai.Client())
job = transcriber.batch_transcribe(file_paths, files_per_request=10)
labels = transcriber.get_batch_results(transcriber.wait_for_batch(job.name))
```

Empty values are the model declining, and those rows are dropped before sampling.


### 2.7 Sample the 3h: `src/prepare_data.py`

From the surviving pool: duration 3 to 15s (mean 6.1s), `pesq_hyp > 2.5` (mean 3.2), and no channel contributing more than 8% of train hours. Then split **by channel**, not by clip.

```python
# src/prepare_data.py
merged = merged[merged["duration"].between(MIN_DURATION_S, MAX_DURATION_S)].copy()
merged["has_cs"] = merged["text"].map(has_latin_codeswitch)

channels = pool["channel"].unique()
rng.shuffle(channels)
hold = set(channels[: max(8, int(0.15 * len(channels)))])    # channels, not clips
silver = subsample_hours(pool[pool["channel"].isin(hold)], SILVER_HOURS, rng)
train = subsample_hours(pool[~pool["channel"].isin(hold)], TRAIN_HOURS, rng,
                        channel_cap=MAX_CHANNEL_SHARE)       # 0.08
leftover = train_pool[~train_pool["id"].isin(set(train["id"]))]
dev = subsample_hours(leftover, DEV_HOURS, rng)
```

The channel cap is enforced while filling the hour budget, so one prolific channel cannot dominate:

```python
# src/prepare_data.py
for row in work.itertuples(index=False):
    if used_by_channel.get(row.channel, 0.0) >= channel_cap * budget_s:
        continue
    kept.append(row)
    used_by_channel[row.channel] = used_by_channel.get(row.channel, 0.0) + row.duration
    total += float(row.duration)
    if total >= budget_s:
        break
```

| split | hours | clips | channels | labels |
| --- | ---: | ---: | ---: | --- |
| train | 3.00 | 1778 | 94 | Gemini |
| validation | 0.15 | 91 | 35 | Gemini |
| silver | 0.35 | 184 | 15 | Gemini, unseen channels |
| AtlasIA gold | 0.18 | 114 | n/a | human |

The channel split matters. I built a random split of the same sizes on the same pool: every silver clip shared a channel with train, across 66 channels. On YouTube data a clip-level random split measures memorised speakers and rooms.

One thing to keep straight: validation and silver are Gemini, so a score there is agreement with the teacher. AtlasIA is human and is the number to report. 114 clips is small; a one-point CER gap is noise.


In [ ]:
import io

# decode=False keeps `datasets` from needing torchcodec; soundfile does the work.
ds = load_dataset(HF_DATASET).cast_column("audio", Audio(decode=False))


def read_audio(entry):
    """Return (mono float32 array, sample rate) from any datasets audio value."""
    if isinstance(entry, dict):
        if entry.get("array") is not None:
            return np.asarray(entry["array"], dtype=np.float32), int(entry["sampling_rate"])
        source = io.BytesIO(entry["bytes"]) if entry.get("bytes") else entry["path"]
        wav, sr = sf.read(source, dtype="float32")
    else:                                    # torchcodec AudioDecoder
        samples = entry.get_all_samples()
        return samples.data.mean(0).numpy().astype(np.float32), int(samples.sample_rate)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    return wav.astype(np.float32), int(sr)


def split_row(name, split):
    dur = [d for d in split["duration"] if d]
    pesq = [p for p in split["pesq_hyp"] if p is not None]
    return {
        "split": name,
        "clips": len(split),
        "hours": round(sum(dur) / 3600, 2),
        "channels": len(set(split["channel"])),
        "mean_s": round(sum(dur) / max(len(dur), 1), 2),
        "code_switch": round(sum(bool(x) for x in split["has_cs"]) / len(split), 2),
        "mean_pesq": round(sum(pesq) / max(len(pesq), 1), 2),
    }

display(pd.DataFrame([split_row(k, ds[k]) for k in ("train", "validation", "silver")]))

# channels really are disjoint
train_ch = set(ds["train"]["channel"])
silver_ch = set(ds["silver"]["channel"])
print("shared channels between train and silver:", len(train_ch & silver_ch))

ex = ds["train"][0]
wav, sr = read_audio(ex["audio"])
print("\nexample:", ex["id"], "|", round(ex["duration"], 2), "s |", ex["channel"],
      "|", sr, "Hz |", wav.shape)
print(ex["text"][:200])


## 3. Scoring

Darija has no standard orthography, so the cleanup is deliberately mild and identical for every model and every split.

Applied: NFKC, strip tashkeel and tatweel, unify alef (`أ إ آ ٱ` to `ا`) and `ى` to `ي`, Arabic-Indic digits to ASCII, drop punctuation except `%`, lowercase.

Not applied: `ة` is kept, because folding it to `ه` hides a real spelling error. Latin is kept, because the labels keep code-switch and deleting it would flatter WER while ignoring French.

I report CER first. WER on Darija largely measures whether two annotators spelled a word the same way. Numbers come with a bootstrap interval, because 91 to 114 clips is small enough that a one point difference means nothing.


In [ ]:
import re
import unicodedata

from jiwer import cer as jiwer_cer, wer as jiwer_wer

_TASHKEEL = re.compile("[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
_ALEF = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا"})
_YEH = str.maketrans({"ى": "ي"})
_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")
_KEEP = re.compile(r"[^\w%]+", flags=re.UNICODE)
_WS = re.compile(r"\s+")


def normalize_text(text):
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    text = _TASHKEEL.sub("", text).replace("\u0640", "")
    text = text.translate(_ALEF).translate(_YEH).translate(_DIGITS)
    return _WS.sub(" ", _KEEP.sub(" ", text)).strip().lower()


def has_latin(text):
    return any("LATIN" in unicodedata.name(c, "") for c in (text or "") if c.isalpha())


def _prep(texts, normalized):
    out = []
    for t in texts:
        s = normalize_text(t) if normalized else str(t or "")
        out.append(s if s.strip() else "<empty>")
    return out


def corpus_scores(refs, hyps, normalized=True):
    r, h = _prep(refs, normalized), _prep(hyps, normalized)
    return {"cer": float(jiwer_cer(r, h)), "wer": float(jiwer_wer(r, h)), "n": len(r)}


def per_utt(refs, hyps, metric="cer", normalized=True):
    fn = jiwer_cer if metric == "cer" else jiwer_wer
    r, h = _prep(refs, normalized), _prep(hyps, normalized)
    return [float(fn([a], [b])) for a, b in zip(r, h)]


def bootstrap_ci(refs, hyps, metric="cer", n_boot=1000, seed=SEED):
    """95% utterance-level bootstrap interval."""
    scores = np.asarray(per_utt(refs, hyps, metric=metric), dtype=np.float64)
    if scores.size == 0:
        return (float("nan"),) * 3
    rng = np.random.default_rng(seed)
    boots = np.array([scores[rng.integers(0, scores.size, scores.size)].mean()
                      for _ in range(n_boot)])
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return float(scores.mean()), float(lo), float(hi)


# sanity check: normalization keeps Latin and ة, drops diacritics
demo = "قَالْ ليه merci بزاف، 100%"
print(repr(normalize_text(demo)))
print(corpus_scores(["واحد جوج", "merci bcp"], ["واحد جوج", "merci beaucoup"]))


## 4. Which model to adapt

Before training anything I decoded [`atlasia/darija-asr-benchmark`](https://huggingface.co/datasets/atlasia/darija-asr-benchmark) (114 clips, 0.18 h, human Darija labels) with stock checkpoints, using the normalization from section 3. The Hub split is named `train`; there is no held-out `test` split.

| model | params M | VRAM MB | RTF | latency ms | CER | WER | Arabic % |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| whisper-tiny | 38 | 596 | 0.016 | 96 | 77.0 | 121.0 | 99.1 |
| whisper-base | 73 | 1105 | 0.023 | 132 | 95.5 | 149.6 | 100.0 |
| whisper-small | 242 | 2096 | 0.045 | 265 | 76.9 | 126.0 | 100.0 |
| whisper-medium | 764 | 4441 | 0.118 | 690 | 64.1 | 97.6 | 100.0 |
| whisper-large-v2 | 1543 | 7280 | 0.212 | 1237 | 70.8 | 111.6 | 100.0 |
| whisper-large-v3 | 1544 | 7269 | 0.155 | 905 | 40.9 | 76.6 | 100.0 |
| whisper-large-v3-turbo | 809 | 3888 | 0.041 | 238 | 40.3 | 79.1 | 100.0 |
| qwen3-asr-0.6b | 782 | 2072 | 0.066 | 384 | 32.4 | 76.8 | 100.0 |
| qwen3-asr-1.7b | 2038 | 4490 | 0.103 | 602 | 28.4 | 71.7 | 100.0 |
| moss-transcribe-0.9b | 908 | 2095 | 0.061 | 354 | 95.7 | 99.0 | 5.3 |
| mms-1b-all | 965 | 5029 | 0.017 | 100 | 35.5 | 85.8 | 100.0 |
| cohere-transcribe | 2066 | 4104 | 0.057 | 334 | 43.8 | 77.6 | 100.0 |
| **cohere-transcribe-arabic** | **2066** | **4103** | **0.078** | **458** | **19.9** | **49.1** | **100.0** |

Reading this:

- Whisper is not a starting point. large-v3 and turbo sit around 40 CER; v2 is worse, not better.
- MOSS is 5% Arabic script, so its CER is not a transcription number. It looks fast because it stops early in the wrong alphabet.
- Qwen 0.6B is usable here (32.4 CER, 100% Arabic). Qwen 1.7B is the strongest LLM-first card at 28.4.
- Multilingual Cohere is 43.8 CER; the Arabic variant of the same 2B model is 19.9. That gap is dialect and code-switch training, not size.

So: adapt `cohere-transcribe-arabic`. It is the strongest stock model in this table. The job is to improve 19.9 without destroying it. n=114 is small; a couple of CER points is noise. Latency was measured on two GPUs, so do not rank families by milliseconds.


### 4.1 Why the encoder-heavy shape wins here

The accuracy column is not the only reason to prefer Cohere over the Qwen and MOSS family. The three models are built the other way round from each other, and for transcription that decides the latency.

The two halves of a seq2seq ASR model do not cost the same:

- The **encoder** runs **once per utterance**, over all frames at the same time. One forward pass, large matmuls, high arithmetic intensity, so the GPU runs near peak. Its cost scales with audio length, not with how much text comes out.
- The **decoder** runs **once per output token**, and each step depends on the previous one. At batch 1 every step is a skinny matrix-vector product, so it is memory-bandwidth bound rather than compute bound and the GPU sits mostly idle. Its cost is paid once per generated token.

So roughly `latency = one encoder pass + T x one decoder step`, where T is the number of generated tokens, around 30 to 60 for a short Darija clip. A parameter in the encoder is paid once at high efficiency. A parameter in the decoder is paid T times at low efficiency. Where the weights sit matters more than how many there are.

Here is where they sit, measured from the checkpoint tensors rather than the model cards. Latency is from the AtlasIA pass (section 4 table).

| model | total | encoder | decoder / LM | decoder share |
| --- | ---: | ---: | ---: | ---: |
| **cohere-transcribe-arabic** | 2066M | **1895M** (48 Conformer layers, d=1280) | **154M** (8 layers, d=1024) | **7%** |
| qwen3-asr-1.7b | 2038M | 317M (24 layers, d=1024) | 1721M (28 layers, d=2048, 152k vocab) | 84% |
| qwen3-asr-0.6b | 782M | 186M | 596M | 76% |
| moss-transcribe-0.9b | 909M | 307M (Whisper encoder) | 596M (28 layers, d=1024) | 66% |
| whisper-large-v3 | 1543M | 637M (32 layers) | 907M (32 layers, 210M of it cross-attention) | 59% |

Cohere puts 92% of its parameters in the part that runs once, and 7% in the part that runs per token. Qwen and MOSS are LLM-first: a Whisper-style audio tower bolted onto a text language model, so most of the capacity is in the part that runs per token.

Cross-attention counts as decoder here, even though it is usually named `encoder_attn`. It reads the encoder output, but it executes inside the decoder once per generated token, so for latency it belongs on the per-token side.

**Cohere against Qwen3-ASR-1.7B is the controlled comparison.** Total size is nearly identical, 2066M against 2038M. Same GPU, same batch 1, same greedy decode. Per generated token, Cohere touches about 150M decoder weights (8 layers at d=1024 plus a small output projection), while Qwen touches about 1.72B (28 layers at d=2048, plus a 2048 x 152k output projection at every single step). That is roughly eleven times more weight traffic per token.

Two failure modes to watch in the table: a model that emits the wrong alphabet and stops early looks fast; a model that over-generates looks slow. A latency number is only meaningful next to a usable transcript (Arabic % near 100, hyp/ref length near 1).

There is a second reason this shape suits a 3 hour budget, which matters for section 5. The 1.9B encoder is where Arabic acoustics live, and it is already good, so freezing it costs nothing and protects the zero-shot quality. The 152M decoder is small enough that 3 hours of data can actually move its spelling behaviour, and cheap to wrap in LoRA. With an LLM-first model, the text side is 1.7B; adapting it on 3 hours of teacher labels is both more expensive and much easier to damage.

Where this argument reverses, to be fair to the alternatives: a 152M decoder has little language modeling capacity, so if the deliverable needed fluent free-form French, rich formatting, or any reasoning over context, the LLM-first models are the better base and Qwen3-ASR-1.7B is the fallback. Encoder cost also grows with audio length while decoder cost grows with output length, so on very long audio with short output the gap narrows. For short-utterance Darija transcription, which is this task, neither applies.


### 4.2 Reproducing the table

The numbers came from the harness in `benchmarks/zeroshot_casablanca/`, pointed at AtlasIA: `lineup.py` holds the model list, `run.py` decodes and measures VRAM, params, RTF and latency per model, `write_report.py` renders the README under `benchmarks/zeroshot_atlasia/`. Per-clip hypotheses land in `benchmarks/zeroshot_atlasia/results/<model>/hyps.jsonl`.

```bash
bash scripts/srun_zeroshot_atlasia.sh                          # all models
bash benchmarks/zeroshot_atlasia/run.sh --only cohere-transcribe-arabic
bash benchmarks/zeroshot_atlasia/run.sh --only whisper-small --max-utts 50
```

The cell below is the same measurement in miniature, for one model at a time. It is off by default because a full pass is a few minutes per model on one GPU. AtlasIA is gated; accept the Hub terms once, then the cell loads split `train`.


In [ ]:
ZERO_SHOT = {
    "whisper-tiny": ("openai/whisper-tiny", "whisper"),
    "whisper-base": ("openai/whisper-base", "whisper"),
    "whisper-small": ("openai/whisper-small", "whisper"),
    "whisper-medium": ("openai/whisper-medium", "whisper"),
    "whisper-large-v2": ("openai/whisper-large-v2", "whisper"),
    "whisper-large-v3": ("openai/whisper-large-v3", "whisper"),
    "whisper-large-v3-turbo": ("openai/whisper-large-v3-turbo", "whisper"),
    "qwen3-asr-0.6b": ("Qwen/Qwen3-ASR-0.6B", "qwen"),
    "qwen3-asr-1.7b": ("Qwen/Qwen3-ASR-1.7B", "qwen"),
    "moss-transcribe-0.9b": ("OpenMOSS-Team/MOSS-Transcribe-Diarize", "moss"),
    "mms-1b-all": ("facebook/mms-1b-all", "ctc"),
    "cohere-transcribe": ("CohereLabs/cohere-transcribe-03-2026", "cohere"),
    "cohere-transcribe-arabic": (BASE_MODEL, "cohere"),
}

RUN_ZEROSHOT = False
ZS_MODEL = "cohere-transcribe-arabic"
ZS_LIMIT = None          # e.g. 50 for a quick check


def load_gold(split="train", limit=None):
    gold = load_dataset(GOLD_DATASET, split=split).cast_column("audio", Audio(decode=False))
    if limit:
        gold = gold.select(range(min(limit, len(gold))))
    return gold


def gold_text(ex):
    for key in ("transcription", "sentence", "text", "transcript"):
        if key in ex and isinstance(ex[key], str):
            return ex[key]
    return ""


def param_split(model):
    """Where the weights sit: encoder runs once per clip, decoder once per token.

    Test for decoder first. Cross-attention is usually named `encoder_attn`, but
    it executes inside the decoder once per generated token, so it belongs on the
    per-token side of the latency budget.
    """
    buckets = {"encoder": 0, "decoder/LM": 0, "other": 0}
    for name, param in model.named_parameters():
        low = name.lower()
        if any(t in low for t in ("decoder", "language_model", "lm_head",
                                  "embed_tokens", "model.layers")):
            key = "decoder/LM"
        elif "encoder" in low or "audio_tower" in low:
            key = "encoder"
        else:
            key = "other"
        buckets[key] += param.numel()
    total = max(sum(buckets.values()), 1)
    out = {k: f"{v / 1e6:.0f}M ({100 * v / total:.0f}%)" for k, v in buckets.items()}
    out["total"] = f"{total / 1e6:.0f}M"
    return out


def load_stock(repo, kind):
    from transformers import AutoProcessor
    processor = AutoProcessor.from_pretrained(repo)
    if kind == "ctc":
        from transformers import AutoModelForCTC
        model = AutoModelForCTC.from_pretrained(repo)
    elif kind == "cohere":
        from transformers import CohereAsrForConditionalGeneration
        model = CohereAsrForConditionalGeneration.from_pretrained(repo, dtype=torch.bfloat16)
    else:
        from transformers import AutoModelForSpeechSeq2Seq
        model = AutoModelForSpeechSeq2Seq.from_pretrained(repo)
    return model.to(DEVICE).eval(), processor


def transcribe(model, processor, kind, wav, sr):
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(torch.as_tensor(wav, dtype=torch.float32),
                                             sr, SAMPLE_RATE).numpy()
    inputs = processor(wav, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items() if hasattr(v, "to")}
    if kind == "ctc":
        with torch.inference_mode():
            logits = model(**inputs).logits
        return processor.batch_decode(logits.argmax(-1))[0].strip()
    if kind == "cohere" and "input_features" in inputs:
        inputs["input_features"] = inputs["input_features"].to(dtype=model.dtype)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=128)
    return processor.batch_decode(out, skip_special_tokens=True)[0].strip()


if RUN_ZEROSHOT:
    repo, kind = ZERO_SHOT[ZS_MODEL]
    if kind in {"qwen", "moss"}:
        raise SystemExit(f"{ZS_MODEL} needs its own package; see the model card")
    model, processor = load_stock(repo, kind)
    print(ZS_MODEL, "parameter split:", param_split(model))
    gold = load_gold("train", ZS_LIMIT)
    refs, hyps, audio_s = [], [], 0.0
    t0 = time.perf_counter()
    for ex in gold:
        wav, sr = read_audio(ex["audio"])
        audio_s += len(wav) / sr
        hyps.append(transcribe(model, processor, kind, wav, sr))
        refs.append(gold_text(ex))
    took = time.perf_counter() - t0
    ref_chars = sum(len(r) for r in refs) / max(len(refs), 1)
    hyp_chars = sum(len(h) for h in hyps) / max(len(hyps), 1)
    print(ZS_MODEL, corpus_scores(refs, hyps),
          "RTF", round(took / max(audio_s, 1e-6), 3),
          "| latency", f"{1000 * took / max(len(refs), 1):.0f}ms",
          # a fast model that emits half the reference length is not fast, it is wrong
          "| hyp/ref chars", round(hyp_chars / max(ref_chars, 1e-6), 2))
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
else:
    print("set RUN_ZEROSHOT = True to reproduce a row")


## 5. Adapting the model

### 5.1 Decisions, and why

The starting point is 20.2 CER on AtlasIA. That single fact drives everything: the risk is not underfitting, it is wrecking a good model with 3 hours of teacher labels.

1. **Freeze the 2B Conformer encoder.** Full finetuning 2B parameters on 3h invites catastrophic forgetting of the Arabic the model already knows, and it does not fit the budget.

2. **LoRA on the decoder** (r=32, alpha=64, dropout 0.05) on q/k/v/o in both self-attention and cross-attention. The decoder is what produces text, so this is where a spelling convention and Latin code-switch are learned. Deliberately not on the MLP or embeddings: with 1778 clips those overfit spelling fast.

3. **A conv adapter inside the encoder**, on layers 15 to 47 (the top two thirds of 48). The Darija mismatch against MSA-heavy training is largely temporal: speech rate, gemination, clipped vowels, short French bursts. That is a job for depthwise convolutions over time, not a low-rank update of attention. Kernels are 7, 15, 23, 31, following MULTI-CONVFORMER (Prabhu et al. 2024); short kernels like 3 or 5 would sit inside the Conformer's own k=9 convolution and add nothing. The up-projection is zero-initialised, so at step 0 the adapter is the identity and the model is exactly the 20.2 CER baseline.

4. **learning rate.** Adapter and decoder LoRA both 1e-4. I did not go to 1e-3; that erased the zero-shot quality in early tries.

5. **AdamW with betas (0.9, 0.98)**, cosine schedule, 10% warmup, 5 epochs, keep the best validation CER. Prompt tokens are masked out of the loss, otherwise the model spends capacity learning to emit its own language tag.

6. **Length-bucketed batches.** Clips run 3 to 15s with a 6.1s mean. A random batch of 32 pads about 49% of the mel frames; sorting into duration windows and shuffling inside them brings that to about 5%. Same compute, roughly twice the useful work.

7. **Augmentation, because train and test differ on purpose.** Train audio is filtered to PESQ > 2.5, AtlasIA is field recordings. Always applied: speed 0.9/1.0/1.1, gain +-3 dB, drop-chunk, SpecAugment. Applied 25% of the time if you point `MUSAN_ROOT` / `RIR_ROOT` at those corpora: additive noise or music, and room impulse responses. Never speech noise, since overlaying other talkers on a Darija clip contradicts the label.

8. **Four recipes over identical data, seed and augmentation.** Only the trainable slice changes, which makes this an ablation rather than four unrelated runs.

| recipe | encoder | decoder | question it answers |
| --- | --- | --- | --- |
| **hybrid** | MultiConv | LoRA | does the combination work |
| `decoder_lora` | frozen | LoRA | how much is just writing convention |
| `full_lora` | LoRA | LoRA | do the convs beat standard PEFT |
| `encoder_lora` | LoRA | frozen | is the gain acoustic |

That comes to 21.1M trainable parameters of 2.09B, spread over 33 encoder adapters and 64 LoRA modules. On disk the hybrid run is still `checkpoints/cohere-method/` (`--recipe method` in the training script).

The rest of this section is the implementation, in order: adapter, LoRA, augmentation, data pipeline, batch sizing, trainer, run. Each part names the repo file it corresponds to. If you only want to launch a run, that is one line:

```bash
bash scripts/srun_train_cohere.sh method          # hybrid: MultiConv + LoRA
# or decoder_lora | full_lora | encoder_lora
```

which requests one GPU, 8 CPUs and 40GB for 3 hours, then calls `python -m src.train_cohere --recipe method` inside the container. `scripts/setup_moss_cohere_env.sh` builds the env if it does not exist yet.


### 5.2 The conv adapter

Repo: `src/adapters.py` (`MultiConvAdapter`, `EncoderBlockWithConvAdapter`, `attach_multiconv_adapters`).

One module per encoder layer, wrapped around the frozen block. Down-project to 64 channels, split in two, use one half as a gate over multi-kernel depthwise convolutions of the other, merge, project back up. Residual, and zero-init on the way out.

The version below is the `concat_fusion` path only. `src/adapters.py` also carries `sum` and `weighted_sum` fusion, which I tried and did not keep.


In [ ]:
import torch.nn as nn


class MultiConvAdapter(nn.Module):
    """Residual multi-kernel conv adapter (MULTI-CONVFORMER style)."""

    def __init__(self, d_model, bottleneck=64, kernels=(7, 15, 23, 31),
                 dropout=0.1, fusion="concat_fusion", merge_kernel=31):
        super().__init__()
        kernels = tuple(int(k) for k in kernels)
        if any(k < 1 or k % 2 == 0 for k in kernels):
            raise ValueError(f"kernels must be odd and positive: {kernels}")
        if bottleneck % len(kernels) != 0:
            raise ValueError("bottleneck must divide by the number of kernels")

        self.kernels = kernels
        self.fusion = fusion
        self.norm = nn.LayerNorm(d_model)
        self.down = nn.Linear(d_model, 2 * bottleneck)   # half signal, half gate
        self.act = nn.GELU()
        self.gate_norm = nn.LayerNorm(bottleneck)

        per = bottleneck // len(kernels)
        self.convs = nn.ModuleList([
            nn.Conv1d(bottleneck, per, kernel_size=k, padding=(k - 1) // 2, groups=per)
            for k in kernels
        ])
        self.merge = (
            nn.Conv1d(bottleneck, bottleneck, kernel_size=merge_kernel,
                      padding=(merge_kernel - 1) // 2, groups=bottleneck)
            if fusion == "concat_fusion" else None
        )
        self.up = nn.Linear(bottleneck, d_model)
        self.drop = nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight)      # identity at step 0
        nn.init.zeros_(self.up.bias)

    def forward(self, hidden_states):           # (batch, time, dim)
        residual = hidden_states
        hidden = self.act(self.down(self.norm(hidden_states)))
        signal, gate = hidden.chunk(2, dim=-1)
        gate = self.gate_norm(gate).transpose(1, 2)
        fused = torch.cat([conv(gate).transpose(1, 2) for conv in self.convs], dim=-1)
        if self.merge is not None:
            fused = fused + self.merge(fused.transpose(1, 2)).transpose(1, 2)
        return residual + self.drop(self.up(signal * fused))


class BlockWithConvAdapter(nn.Module):
    """Wrap a frozen encoder block so the adapter sees its output."""

    def __init__(self, block, adapter):
        super().__init__()
        self.block = block
        self.conv_adapter = adapter

    def forward(self, *args, **kwargs):
        out = self.block(*args, **kwargs)
        if isinstance(out, tuple):
            return (self.conv_adapter(out[0]),) + out[1:]
        return self.conv_adapter(out)


def get_encoder(model):
    core = model.get_base_model() if hasattr(model, "get_base_model") else model
    if hasattr(core, "model") and hasattr(core.model, "encoder"):
        return core.model.encoder
    return core.encoder


def attach_conv_adapters(model, cfg=ADAPTER):
    """Insert one adapter per encoder layer above skip_bottom_frac."""
    encoder = get_encoder(model)
    layers = encoder.layers
    start = int(len(layers) * cfg["skip_bottom_frac"])
    d_model = int(getattr(encoder.config, "hidden_size", 1280))
    attached = []
    for idx in range(start, len(layers)):
        block = layers[idx]
        if isinstance(block, BlockWithConvAdapter):
            continue
        adapter = MultiConvAdapter(
            d_model,
            bottleneck=cfg["bottleneck"], kernels=cfg["kernels"],
            dropout=cfg["dropout"], fusion=cfg["fusion"],
            merge_kernel=cfg["merge_kernel"],
        )
        ref = next(block.parameters())
        adapter.to(device=ref.device, dtype=ref.dtype)
        layers[idx] = BlockWithConvAdapter(block, adapter)
        attached.append(idx)
    return {"n_layers": len(layers), "start_layer": start,
            "attached_layers": attached, "d_model": d_model, **{k: cfg[k] for k in cfg}}


# quick shape and identity check on CPU
probe = MultiConvAdapter(64, bottleneck=16, kernels=(3, 5))
x = torch.randn(2, 30, 64)
print("shape kept:", probe(x).shape, "| identity at init:", torch.allclose(probe(x), x))
print("adapter params per layer (d=1280):",
      sum(p.numel() for p in MultiConvAdapter(1280, **{k: ADAPTER[k] for k in
          ("bottleneck", "kernels", "dropout", "fusion", "merge_kernel")}).parameters()))


### 5.3 LoRA placement

Repo: `src/adapters.py` (`lora_targets_for_scope`) and `src/cohere_runtime.py` (`attach_decoder_lora`).

`target_kinds` picks the projections; the scope decides which side of the model they come from. The decoder gets self-attention and cross-attention, the encoder only self-attention. Nothing touches MLP or embeddings.

PEFT's `TaskType.SEQ_2_SEQ_LM` wrapper does not work with this architecture: it always calls the base model with `input_ids`, and `CohereAsr` then passes `input_ids` to its decoder as well. Building `LoraConfig` without a task type avoids that. When the conv adapter is present it goes in `modules_to_save` so it trains and gets serialised with the LoRA weights.


In [ ]:
from peft import LoraConfig, get_peft_model


def _in_decoder(name):
    return ".decoder." in f".{name}." or name.startswith("decoder.")


def lora_targets(model, scope, kinds=LORA["target_kinds"]):
    names = []
    for name, _ in model.named_modules():
        if name.rsplit(".", 1)[-1] not in kinds:
            continue
        dec = _in_decoder(name)
        if scope == "decoder" and not dec:
            continue
        if scope == "encoder" and dec:
            continue
        if dec and ("self_attn" not in name and "encoder_attn" not in name):
            continue      # skip decoder MLP-adjacent projections
        if not dec and "self_attn" not in name:
            continue      # encoder: attention only
        names.append(name)
    if not names:
        raise RuntimeError(f"no LoRA targets found for scope={scope}")
    return names


def attach_lora(model, scope):
    targets = lora_targets(model, scope)
    has_conv = any(isinstance(getattr(m, "conv_adapter", None), nn.Module)
                   for m in model.modules())
    kwargs = dict(
        r=LORA["r"], lora_alpha=LORA["lora_alpha"],
        lora_dropout=LORA["lora_dropout"], bias=LORA["bias"],
        target_modules=targets,
    )
    if has_conv:
        kwargs["modules_to_save"] = ["conv_adapter"]
    model = get_peft_model(model, LoraConfig(**kwargs))

    # PEFT clones modules_to_save onto CPU; move them back and unfreeze.
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    for module in model.modules():
        for attr in ("conv_adapter", "modules_to_save"):
            sub = getattr(module, attr, None)
            if isinstance(sub, nn.Module):
                sub.to(device=device, dtype=dtype)
    for name, param in model.named_parameters():
        if "conv_adapter" in name:
            param.requires_grad = True
    return model, targets


def trainable_summary(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return {"trainable": trainable, "total": total,
            "pct": round(100 * trainable / max(total, 1), 3)}

print("LoRA helpers ready")


### 5.4 Augmentation

Repo: `src/augment.py` (`WaveformAugment`, `spec_augment_mel`).

Waveform first (speed, gain, reverb, noise, drop-chunk), then SpecAugment on the mel features. Noise and reverb only fire if the corpora are available; file lists are cached to disk so the first epoch does not walk the tree repeatedly.

One trap worth knowing about: speed perturbation by resampling has to keep the rate ratio rational, or `torchaudio` builds a kernel with one row per output frequency and blows up memory. The helper below explains the arithmetic. `src/augment.py` prefers SpeechBrain's `SpeedPerturb` and `DropChunk` when they are installed, and falls back to this same torch path when they are not.


In [ ]:
AUDIO_EXTS = {".wav", ".flac", ".ogg"}


def cached_file_list(root, cache_path, exclude=()):
    cache_path = Path(cache_path)
    if cache_path.is_file():
        return [l for l in cache_path.read_text().splitlines() if l]
    root = Path(root)
    paths = []
    if root.is_dir():
        for dirpath, _, files in os.walk(root):
            for name in files:
                if Path(name).suffix.lower() in AUDIO_EXTS and not any(
                        tok in name.lower() for tok in exclude):
                    paths.append(str(Path(dirpath) / name))
    paths.sort()
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_text("\n".join(paths))
    return paths


def load_mono(path, sample_rate=SAMPLE_RATE):
    wav, sr = torchaudio.load(path)
    wav = wav.float().mean(dim=0)
    if int(sr) != sample_rate:
        wav = torchaudio.functional.resample(wav, int(sr), sample_rate)
    return wav


def mix_at_snr(wav, noise, snr_db):
    n = wav.numel()
    if noise.numel() < n:
        noise = noise.repeat((n + noise.numel() - 1) // noise.numel())
    start = int(torch.randint(0, max(noise.numel() - n, 1), (1,)).item())
    noise = noise[start:start + n]
    power = wav.pow(2).mean().clamp_min(1e-8)
    npow = noise.pow(2).mean().clamp_min(1e-8)
    return wav + noise * torch.sqrt(power / (npow * 10.0 ** (snr_db / 10.0)))


def convolve_rir(wav, rir):
    rir = rir / rir.abs().max().clamp_min(1e-8)
    mixed = torchaudio.functional.fftconvolve(wav, rir, mode="full")
    peak = int(rir.abs().argmax().item())
    out = mixed[peak:peak + wav.numel()]
    if out.numel() < wav.numel():
        out = torch.nn.functional.pad(out, (0, wav.numel() - out.numel()))
    src = wav.pow(2).mean().sqrt().clamp_min(1e-8)
    cur = out.pow(2).mean().sqrt().clamp_min(1e-8)
    return out * (src / cur)


def speed_perturb(wav, factor, sample_rate=SAMPLE_RATE):
    """Resample to a rational multiple of the rate.

    Use rate * factor and snap to 100 Hz. Dividing instead (16000 / 0.9 = 17778)
    leaves gcd(16000, 17778) = 2, and torchaudio then builds an 8889 x 96001
    kernel, which is several GB. 16000 * 0.9 = 14400 shares gcd 1600, so the
    kernel is 9 x 120.
    """
    if abs(factor - 1.0) < 1e-6:
        return wav
    new_freq = int(round(sample_rate * factor / 100.0)) * 100
    return torchaudio.functional.resample(wav, sample_rate, max(new_freq, 1000))


class WaveformAugment:
    def __init__(self, cfg=AUG, musan_root=MUSAN_ROOT, rir_root=RIR_ROOT,
                 cache_dir=None, seed=SEED):
        self.cfg = cfg
        self.rng = np.random.default_rng(seed)
        cache = Path(cache_dir or (WORK_DIR / "cache"))
        self.noises, self.musics, self.rirs = [], [], []
        if musan_root:
            self.noises = cached_file_list(Path(musan_root) / "noise", cache / "musan_noise.txt")
            self.musics = cached_file_list(Path(musan_root) / "music", cache / "musan_music.txt")
        if rir_root:
            root = Path(rir_root)
            self.rirs = (cached_file_list(root / "simulated_rirs", cache / "rir_sim.txt")
                         + cached_file_list(root / "real_rirs_isotropic_noises",
                                            cache / "rir_real.txt", exclude=("noise",)))
        print(f"[aug] musan noise={len(self.noises)} music={len(self.musics)} rir={len(self.rirs)}")

    def _pick(self, pool):
        return pool[int(self.rng.integers(0, len(pool)))] if pool else None

    def __call__(self, wav):
        cfg = self.cfg
        wav = wav.float()

        wav = speed_perturb(wav, float(self.rng.choice(cfg["speed_factors"])))
        wav = wav * (10.0 ** (float(self.rng.uniform(-cfg["gain_db"], cfg["gain_db"])) / 20.0))

        if self.rirs and self.rng.random() < cfg["rir_prob"]:
            try:
                wav = convolve_rir(wav, load_mono(self._pick(self.rirs)))
            except Exception:
                pass

        if (self.noises or self.musics) and self.rng.random() < cfg["noise_prob"]:
            pool = self.musics if (self.musics and self.rng.random() < 0.2) else self.noises
            try:
                snr = float(self.rng.uniform(*cfg["noise_snr"]))
                wav = mix_at_snr(wav, load_mono(self._pick(pool)), snr)
            except Exception:
                pass

        if self.rng.random() < cfg["drop_chunk_prob"] and wav.numel() > 400:
            n = wav.numel()
            width = max(1, int(self.rng.uniform(0.02, cfg["drop_chunk_max_frac"]) * n))
            start = int(self.rng.integers(0, max(n - width, 1)))
            wav = wav.clone()
            wav[start:start + width] = 0
        return wav


def spec_augment(features, rng, cfg=AUG):
    """features: (time, n_mels). Zero out time and frequency bands."""
    out = features.clone()
    n_frames, n_mels = out.shape
    for _ in range(cfg["specaug_n_time"]):
        width = max(1, int(cfg["specaug_time_frac"] * n_frames))
        start = int(rng.integers(0, max(n_frames - width, 1)))
        out[start:start + width, :] = 0
    for _ in range(cfg["specaug_n_freq"]):
        width = max(1, int(cfg["specaug_freq_frac"] * n_mels))
        start = int(rng.integers(0, max(n_mels - width, 1)))
        out[:, start:start + width] = 0
    return out


wave_aug = WaveformAugment()
_demo = torch.randn(SAMPLE_RATE * 4) * 0.05
for _f in AUG["speed_factors"]:
    _t = time.perf_counter()
    _n = speed_perturb(_demo, _f).numel()
    print(f"speed {_f}: {_demo.numel()} -> {_n} samples in {time.perf_counter() - _t:.3f}s")
print("full chain:", wave_aug(_demo).numel(), "samples")


### 5.5 Data pipeline

Repo: `src/train_cohere.py` (`CohereJsonlDataset`, `LengthBucketSampler`, `CohereCollator`). The only real difference is the source: the script reads the local `data/splits/*.jsonl` manifests written by `src/prepare_data.py`, while the notebook streams the same clips from the Hub.

The dataset streams straight from the Hub. Labels are `[prompt ids] + [text ids] + [eos]`, and the collator sets the prompt positions to -100 so the loss only covers the transcript. The sampler sorts by duration, cuts windows of `mix * batch`, shuffles inside each window and then shuffles the batches, which keeps randomness while putting similar lengths together.


In [ ]:
from torch.nn.utils.rnn import pad_sequence


class HubAsrDataset(torch.utils.data.Dataset):
    def __init__(self, split, processor, augment=False, seed=SEED):
        self.rows = load_dataset(HF_DATASET, split=split).cast_column(
            "audio", Audio(decode=False))
        self.durations = [float(d or 0.0) for d in self.rows["duration"]]
        self.processor = processor
        self.prompt_ids = list(processor.get_decoder_prompt_ids(
            language=LANGUAGE, punctuation=True))
        self.eos_id = processor.tokenizer.eos_token_id
        self.augment = augment
        self.rng = np.random.default_rng(seed)
        self.wave_aug = wave_aug if augment else None

    def __len__(self):
        return len(self.rows)

    def _labels(self, text):
        ids = list(self.prompt_ids) + list(
            self.processor.tokenizer(text, add_special_tokens=False).input_ids)
        if self.eos_id is not None and (not ids or ids[-1] != self.eos_id):
            ids.append(self.eos_id)
        return ids

    def __getitem__(self, idx):
        ex = self.rows[idx]
        array, sr = read_audio(ex["audio"])
        wav = torch.from_numpy(array)
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
        if self.wave_aug is not None:
            wav = self.wave_aug(wav)

        feats = self.processor.feature_extractor(
            wav.numpy(), sampling_rate=SAMPLE_RATE,
            return_tensors="pt", return_attention_mask=True)
        input_features = feats["input_features"].squeeze(0)
        attention_mask = feats["attention_mask"].squeeze(0)
        if self.augment:
            input_features = spec_augment(input_features, self.rng)
            input_features = input_features * attention_mask.to(input_features.dtype).unsqueeze(-1)

        return {
            "input_features": input_features,
            "attention_mask": attention_mask,
            "labels": torch.tensor(self._labels(ex["text"]), dtype=torch.long),
            "decoder_prompt_ids": torch.tensor(self.prompt_ids, dtype=torch.long),
            "audio_frames": int(attention_mask.sum().item()),
        }


class LengthBucketSampler(torch.utils.data.Sampler):
    def __init__(self, lengths, batch_size, mix=4, seed=SEED):
        self.lengths = np.asarray(lengths, dtype=np.float64)
        self.batch_size = max(int(batch_size), 1)
        self.mix = max(int(mix), 1)
        self.seed = int(seed)
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return int(self.lengths.size)

    def batches(self, epoch=0):
        rng = np.random.default_rng(self.seed + epoch)
        order = np.argsort(self.lengths, kind="stable")
        window = self.batch_size * self.mix
        out = []
        for start in range(0, len(order), window):
            chunk = order[start:start + window].copy()
            rng.shuffle(chunk)
            out += [chunk[i:i + self.batch_size].tolist()
                    for i in range(0, len(chunk), self.batch_size)]
        rng.shuffle(out)
        return [b for b in out if b]

    def __iter__(self):
        for batch in self.batches(self.epoch):
            yield from batch


def shift_tokens_right(input_ids, pad_id, start_id):
    shifted = input_ids.new_zeros(input_ids.shape)
    shifted[:, 1:] = input_ids[:, :-1].clone()
    shifted[:, 0] = start_id
    shifted.masked_fill_(shifted == -100, pad_id)
    return shifted


@dataclass
class CohereCollator:
    pad_id: int
    start_id: int
    pad_fracs: list = None

    def __call__(self, features):
        feats = [f["input_features"] for f in features]
        masks = [f["attention_mask"] for f in features]
        labels = [f["labels"] for f in features]
        prompts = [f["decoder_prompt_ids"] for f in features]

        max_t = max(f.size(0) for f in feats)
        packed = feats[0].new_zeros(len(feats), max_t, feats[0].size(-1))
        att = torch.zeros(len(feats), max_t, dtype=torch.long)
        for i, (feat, mask) in enumerate(zip(feats, masks)):
            t = feat.size(0)
            packed[i, :t] = feat
            att[i, :t] = mask[:t].long()
        if self.pad_fracs is not None and max_t:
            frames = [int(f["audio_frames"]) for f in features]
            self.pad_fracs.append(1.0 - sum(frames) / (len(frames) * max_t))

        labels_pad = pad_sequence(labels, batch_first=True, padding_value=-100)
        decoder_input_ids = shift_tokens_right(
            labels_pad.masked_fill(labels_pad == -100, self.pad_id),
            self.pad_id, self.start_id)
        labels_pad[:, :prompts[0].numel()] = -100      # train text, not the prompt

        return {
            "input_features": packed,
            "attention_mask": att,
            "labels": labels_pad,
            "decoder_input_ids": decoder_input_ids,
            "decoder_prompt_ids": pad_sequence(prompts, batch_first=True,
                                               padding_value=self.pad_id),
        }


def pad_waste(durations, batches):
    out = []
    for batch in batches:
        vals = [max(durations[i], 1e-3) for i in batch]
        out.append(1.0 - sum(vals) / (len(vals) * max(vals)))
    return float(np.mean(out)) if out else 0.0


# what bucketing actually buys, measured on the real durations
durations = [float(d or 0.0) for d in ds["train"]["duration"]]
rng = np.random.default_rng(SEED)
shuffled = rng.permutation(len(durations)).tolist()
random_batches = [shuffled[i:i + 32] for i in range(0, len(shuffled), 32)]
bucketed = LengthBucketSampler(durations, batch_size=32).batches(0)
print(f"pad waste at batch 32: random {pad_waste(durations, random_batches):.1%} "
      f"| bucketed {pad_waste(durations, bucketed):.1%}")


### 5.6 Batch size against the VRAM budget

Repo: `src/train_cohere.py` (`choose_train_batch`, `gpu_mem_gb`).

Gradient checkpointing plus a tiny trainable slice makes activations cheap, so the way to fill the card is a bigger micro-batch rather than more accumulation. The probe runs a real forward and backward on the longest clips and grows the batch until reserved memory reaches ~82% of the budget, leaving headroom for generation during eval.


In [ ]:
# keys CohereAsr.forward accepts. PEFT and the Trainer add extras that break it.
FORWARD_KEYS = {"input_features", "attention_mask", "decoder_input_ids",
                "decoder_attention_mask", "encoder_outputs", "past_key_values",
                "labels", "use_cache"}


def gpu_mem_gb():
    if not torch.cuda.is_available():
        return {"alloc": 0.0, "reserved": 0.0, "peak": 0.0, "device_used": 0.0, "device_total": 0.0}
    free, total = torch.cuda.mem_get_info()
    return {
        "alloc": torch.cuda.memory_allocated() / 1024**3,
        "reserved": torch.cuda.memory_reserved() / 1024**3,
        "peak": torch.cuda.max_memory_allocated() / 1024**3,
        "device_used": (total - free) / 1024**3,
        "device_total": total / 1024**3,
    }


def to_device(batch, model):
    dtype = next(model.parameters()).dtype
    out = {}
    for key, value in batch.items():
        if not torch.is_tensor(value):
            continue
        out[key] = (value.to(device=DEVICE, dtype=dtype)
                    if value.is_floating_point() else value.to(DEVICE))
    return {k: v for k, v in out.items() if k in FORWARD_KEYS}


def choose_train_batch(model, dataset, collator, budget_gb=VRAM_BUDGET_GB,
                       start=8, step=8, max_bs=128, target_frac=0.82):
    if DEVICE != "cuda":
        return start
    target = budget_gb * target_frac
    longest = sorted(range(len(dataset)), key=lambda i: dataset.durations[i], reverse=True)
    best = start
    batch_size = start
    model.train()
    while batch_size <= min(max_bs, len(dataset)):
        try:
            batch = to_device(collator([dataset[i] for i in longest[:batch_size]]), model)
            model.zero_grad(set_to_none=True)
            torch.cuda.reset_peak_memory_stats()
            outputs = model(**batch)
            (outputs["loss"] if isinstance(outputs, dict) else outputs.loss).backward()
            mem = gpu_mem_gb()
            used = max(mem["reserved"], mem["peak"], mem["device_used"])
            print(f"[probe] bs={batch_size} used={used:.2f}GB target={target:.2f}GB")
            model.zero_grad(set_to_none=True)
            del outputs, batch
            torch.cuda.empty_cache()
            if used > target:
                break
            best = batch_size
            batch_size += step
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            print(f"[probe] OOM at bs={batch_size}, keeping {best}")
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            break
    print(f"[probe] chose train batch {best}")
    return best

print("probe ready")


### 5.7 Trainer

Repo: `src/train_cohere.py` (`CohereTrainer`, `LengthBucketCallback`, `VramMonitor`) and `src/cohere_runtime.py` (`load_cohere_base`, `save_student`).

Three things need overriding. The optimizer must put adapter and LoRA parameters in separate groups so they get separate learning rates. `CohereAsr` rejects the extra keys PEFT and the Trainer add, so inputs are filtered. And evaluation needs the decoder prompt for `generate` while the loss needs the full teacher-forced sequence, so `prediction_step` handles them separately instead of passing both.


In [ ]:
from transformers import (AutoProcessor, CohereAsrForConditionalGeneration,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback)


def clean_inputs(inputs, model):
    out = {k: v for k, v in inputs.items() if k in FORWARD_KEYS}
    try:
        dtype = next(model.parameters()).dtype
    except StopIteration:
        return out
    feats = out.get("input_features")
    if torch.is_tensor(feats) and feats.is_floating_point() and feats.dtype != dtype:
        out["input_features"] = feats.to(dtype=dtype)
    return out


class CohereTrainer(Seq2SeqTrainer):
    def _get_train_sampler(self, train_dataset=None):
        dataset = train_dataset if train_dataset is not None else self.train_dataset
        return LengthBucketSampler(dataset.durations, batch_size=self._train_batch_size,
                                   mix=4, seed=self.args.seed)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**clean_inputs(dict(inputs), model))
        loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self):
        if getattr(self, "optimizer", None) is not None:
            return self.optimizer
        adapter, lora, other = [], [], []
        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            in_dec = ".decoder." in f".{name}." or name.startswith("decoder.")
            if "conv_adapter" in name or ("lora_" in name and not in_dec):
                adapter.append(param)          # encoder side: slow
            elif "lora_" in name:
                lora.append(param)             # decoder side: faster
            else:
                other.append(param)
        groups = []
        if adapter:
            groups.append({"params": adapter, "lr": LR_ADAPTER, "weight_decay": 0.01})
        if lora:
            groups.append({"params": lora, "lr": LR_LORA, "weight_decay": 0.0})
        if other:
            groups.append({"params": other, "lr": LR_LORA, "weight_decay": 0.01})
        if not groups:
            raise RuntimeError("nothing to train")
        try:
            self.optimizer = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-8,
                                               fused=torch.cuda.is_available())
        except TypeError:
            self.optimizer = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-8)
        return self.optimizer

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None, **gen_kwargs):
        inputs = dict(inputs)
        prompt = inputs.pop("decoder_prompt_ids", None)
        clean = clean_inputs(inputs, model)
        if prediction_loss_only or not self.args.predict_with_generate:
            return super().prediction_step(model, clean, prediction_loss_only,
                                           ignore_keys=ignore_keys)
        clean = self._prepare_inputs(clean)
        if prompt is not None:
            prompt = self._prepare_inputs({"decoder_input_ids": prompt})["decoder_input_ids"]
        gen = {"input_features": clean["input_features"],
               "attention_mask": clean["attention_mask"]}
        if prompt is not None:
            gen["decoder_input_ids"] = prompt
        with torch.no_grad():
            generated = model.generate(**gen, max_new_tokens=MAX_NEW_TOKENS)
            loss = None
            if "labels" in clean:
                outputs = model(**{k: clean[k] for k in FORWARD_KEYS if k in clean})
                loss = (outputs["loss"] if isinstance(outputs, dict)
                        else outputs.loss).detach().mean()

        # The Trainer concatenates predictions across eval steps, so every
        # batch has to come back the same width.
        labels = clean.get("labels")
        width = max(MAX_NEW_TOKENS + 1, generated.shape[-1],
                    labels.shape[-1] if labels is not None else 0)
        if generated.shape[-1] < width:
            generated = self._pad_tensors_to_max_len(generated, width)
        if labels is not None and labels.shape[-1] < width:
            labels = self._pad_tensors_to_max_len(labels, width)
        return loss, generated, labels


class EpochBuckets(TrainerCallback):
    """Reshuffle buckets each epoch and report real mel padding."""

    def __init__(self, pad_fracs):
        self.pad_fracs = pad_fracs
        self.seen = 0

    def on_epoch_begin(self, args, state, control, train_dataloader=None, **kwargs):
        sampler = getattr(train_dataloader, "sampler", None)
        if sampler is not None and hasattr(sampler, "set_epoch"):
            sampler.set_epoch(int(state.epoch or 0))

    def on_log(self, args, state, control, logs=None, **kwargs):
        if len(self.pad_fracs) > self.seen:
            chunk = self.pad_fracs[self.seen:]
            self.seen = len(self.pad_fracs)
            print(f"[pad] step={state.global_step} batch={np.mean(chunk):.1%} "
                  f"running={np.mean(self.pad_fracs):.1%}")


class VramMonitor(TrainerCallback):
    def __init__(self, budget_gb=VRAM_BUDGET_GB):
        self.budget = budget_gb
        self.peak = 0.0

    def _log(self, tag, state=None):
        mem = gpu_mem_gb()
        used = max(mem["reserved"], mem["peak"], mem["device_used"])
        self.peak = max(self.peak, used)
        step = getattr(state, "global_step", None)
        print(f"[vram:{tag}] step={step} used={used:.2f}/{self.budget:.0f}GB "
              f"({100 * used / self.budget:.0f}%)")

    def on_train_begin(self, args, state, control, **kwargs):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        self._log("weights", state)

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        self._log("eval", state)
        if metrics:
            print({k: round(v, 4) for k, v in metrics.items()
                   if any(t in k for t in ("cer", "wer", "loss"))})

    def on_train_end(self, args, state, control, **kwargs):
        print(f"[vram] peak {self.peak:.2f}GB of {self.budget:.0f}GB")


def build_student(recipe_name=RECIPE):
    recipe = RECIPES[recipe_name]
    processor = AutoProcessor.from_pretrained(BASE_MODEL)
    model = CohereAsrForConditionalGeneration.from_pretrained(BASE_MODEL, dtype=torch.bfloat16)
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model.to(DEVICE)
    meta = attach_conv_adapters(model) if recipe["conv_adapter"] else {"attached_layers": []}
    model, targets = attach_lora(model, recipe["lora_scope"])
    counts = trainable_summary(model)
    print(f"recipe={recipe_name} encoder_adapters={len(meta['attached_layers'])} "
          f"lora_modules={len(targets)} trainable={counts['trainable']:,} "
          f"({counts['pct']}% of {counts['total']:,})")
    return model, processor, meta, targets, counts

print("trainer ready")


### 5.8 Run it

Repo: `src/train_cohere.py` (`main`) and `scripts/srun_train_cohere.sh`.

Set `RUN_TRAIN = True`. On a 16GB card with the defaults this is roughly 280 steps and a couple of hours. The best checkpoint by validation CER is saved to `runs/cohere-<recipe>/best` together with a short record of the settings used.

For the real runs I use the script instead, four jobs over the same data and seed:

```bash
bash scripts/srun_train_cohere.sh method          # hybrid: MultiConv + LoRA
bash scripts/srun_train_cohere.sh decoder_lora
bash scripts/srun_train_cohere.sh full_lora
bash scripts/srun_train_cohere.sh encoder_lora

# smoke test first: 20 steps, fails fast if anything is wrong
python -m src.train_cohere --recipe method --max-steps 20
```

Those write to `checkpoints/cohere-<recipe>/` (hybrid lands in `cohere-method/`), and alongside the weights you get `run_meta.json` (every setting, plus measured padding waste and VRAM peak), `DECISIONS.txt` (the recipe in plain text) and `vram.jsonl` (memory at every log and eval step).


In [ ]:
RUN_TRAIN = False
USE_VRAM_PROBE = True

if RUN_TRAIN:
    model, processor, meta, targets, counts = build_student(RECIPE)
    pad_id = processor.tokenizer.pad_token_id
    start_id = model.config.decoder_start_token_id or processor.get_decoder_prompt_ids(
        language=LANGUAGE, punctuation=True)[0]

    train_ds = HubAsrDataset("train", processor, augment=True)
    dev_ds = HubAsrDataset("validation", processor, augment=False)
    pad_fracs = []
    collator = CohereCollator(pad_id, start_id, pad_fracs)

    train_bs = TRAIN_BS
    if USE_VRAM_PROBE:
        train_bs = choose_train_batch(model, train_ds, CohereCollator(pad_id, start_id))
    accum = 1 if train_bs >= 16 else GRAD_ACCUM

    def compute_metrics(pred):
        pred_ids = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
        pred_ids = np.asarray(pred_ids)
        if pred_ids.ndim == 3:
            pred_ids = pred_ids.argmax(-1)
        safe = pad_id if pad_id is not None else 0
        pred_ids = np.where(pred_ids < 0, safe, pred_ids).astype(np.int64)
        label_ids = np.where(pred.label_ids != -100, pred.label_ids, safe).astype(np.int64)
        hyps = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        refs = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
        raw = corpus_scores(refs, hyps, normalized=False)
        norm = corpus_scores(refs, hyps, normalized=True)
        return {"cer": norm["cer"], "wer": norm["wer"],
                "cer_raw": raw["cer"], "wer_raw": raw["wer"]}

    out_dir = WORK_DIR / f"cohere-{RECIPE}"
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    args = Seq2SeqTrainingArguments(
        output_dir=str(out_dir),
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=EVAL_BS,
        gradient_accumulation_steps=accum,
        learning_rate=LR_LORA,          # per-group LRs are set in create_optimizer
        warmup_steps=0.1,               # transformers 5: a float below 1 is a ratio
        lr_scheduler_type="cosine",
        num_train_epochs=EPOCHS,
        max_steps=MAX_STEPS,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        logging_steps=10,
        predict_with_generate=True,
        generation_max_length=MAX_NEW_TOKENS,
        bf16=use_bf16,
        bf16_full_eval=use_bf16,
        fp16=not use_bf16 and torch.cuda.is_available(),
        gradient_checkpointing=True,
        max_grad_norm=1.0,
        weight_decay=0.01,
        report_to=[],
        load_best_model_at_end=True,
        metric_for_best_model="cer",
        greater_is_better=False,
        save_total_limit=2,
        seed=SEED,
        remove_unused_columns=False,
        label_names=["labels"],
        dataloader_num_workers=NUM_WORKERS,
    )

    trainer = CohereTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=dev_ds,
        data_collator=collator, compute_metrics=compute_metrics,
        callbacks=[EpochBuckets(pad_fracs), VramMonitor()],
        processing_class=processor,
    )
    print(f"train clips={len(train_ds)} dev={len(dev_ds)} batch={train_bs} "
          f"accum={accum} effective={train_bs * accum}")
    trainer.train()

    best = out_dir / "best"
    best.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(best)
    processor.save_pretrained(best)
    torch.save({k: v.detach().cpu() for k, v in trainer.model.state_dict().items()
                if "conv_adapter" in k}, best / "encoder_adapters.pt")
    (best / "run_meta.json").write_text(json.dumps({
        "base_model": BASE_MODEL, "recipe": RECIPE, "dataset": HF_DATASET,
        "lora": {k: v for k, v in LORA.items() if k != "target_kinds"},
        "lora_targets": len(targets),
        "adapter": {**ADAPTER, "kernels": list(ADAPTER["kernels"])},
        "attached_layers": meta["attached_layers"], "trainable": counts,
        "lr_adapter": LR_ADAPTER, "lr_lora": LR_LORA, "epochs": EPOCHS,
        "train_batch": train_bs, "grad_accum": accum, "seed": SEED,
        "mel_pad_waste": float(np.mean(pad_fracs)) if pad_fracs else None,
    }, indent=2))
    print("saved", best)
else:
    print("set RUN_TRAIN = True to train")


## 6. Evaluation

Three sets, three different meanings:

- **validation** (91 clips, Gemini): the model selection signal. Low CER here means it matched the teacher.
- **silver** (184 clips, Gemini, channels never seen in training): does it hold up on new speakers and rooms, still measured against the teacher.
- **AtlasIA** (114 clips, human): the number to report.

Every score comes with a bootstrap interval, and AtlasIA is also broken down by clip length and code-switch, because a single corpus average hides where a Darija model actually fails. 114 clips is small; treat a one-point CER gap as noise.

Repo: `scripts/eval_gold.py` with `--manifest data/splits/gold_atlasia.jsonl`, launched with `scripts/srun_eval_atlasia.sh`.

```bash
bash scripts/srun_eval_atlasia.sh method          # hybrid checkpoint
bash scripts/srun_eval_atlasia.sh zeroshot
python scripts/eval_gold.py --model checkpoints/cohere-method/best --name hybrid \
       --manifest data/splits/gold_atlasia.jsonl --out checkpoints/atlasia_eval/method.json
```

The cells below are the short version: decode, score, slice.


In [ ]:
def load_student(path):
    """Reload a trained checkpoint: base model, conv adapters, then LoRA."""
    path = Path(path)
    meta_path = path / "run_meta.json"
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    processor = AutoProcessor.from_pretrained(path if (path / "preprocessor_config.json").exists()
                                              else BASE_MODEL)
    model = CohereAsrForConditionalGeneration.from_pretrained(
        meta.get("base_model", BASE_MODEL), dtype=torch.bfloat16)
    if meta.get("attached_layers"):
        cfg = dict(ADAPTER)
        cfg.update(meta.get("adapter", {}))
        attach_conv_adapters(model, cfg)
    if (path / "adapter_config.json").exists():
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, str(path))
    extra = path / "encoder_adapters.pt"
    if extra.exists():
        model.load_state_dict(torch.load(extra, map_location="cpu"), strict=False)
    return model.to(DEVICE).eval(), processor


@torch.inference_mode()
def transcribe_cohere(model, processor, wav, sr=SAMPLE_RATE):
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(torch.as_tensor(wav, dtype=torch.float32),
                                             sr, SAMPLE_RATE).numpy()
    inputs = processor(np.asarray(wav, dtype=np.float32), sampling_rate=SAMPLE_RATE,
                       return_tensors="pt", language=LANGUAGE)
    try:
        inputs = inputs.to(DEVICE, dtype=model.dtype)
    except (AttributeError, TypeError):
        inputs = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in inputs.items()}
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    # the Cohere processor needs the chunk index to stitch long audio back together
    chunk = inputs.get("audio_chunk_index") if hasattr(inputs, "get") else None
    try:
        text = processor.decode(out, skip_special_tokens=True,
                                audio_chunk_index=chunk, language=LANGUAGE)
        if isinstance(text, (list, tuple)):
            text = text[0]
    except TypeError:
        text = processor.batch_decode(out, skip_special_tokens=True)[0]
    return str(text).strip()


def evaluate_split(model, processor, rows, tag, text_key="text"):
    """rows: iterable of dicts with audio + reference text."""
    refs, hyps, meta, audio_s = [], [], [], 0.0
    t0 = time.perf_counter()
    for ex in rows:
        wav, sr = read_audio(ex["audio"])
        audio_s += len(wav) / sr
        hyps.append(transcribe_cohere(model, processor, wav, sr))
        refs.append(ex.get(text_key) or gold_text(ex))
        meta.append({
            "duration": float(ex.get("duration") or len(wav) / sr),
            "has_cs": bool(ex.get("has_cs")) if ex.get("has_cs") is not None
                      else has_latin(refs[-1]),
        })
    took = time.perf_counter() - t0
    cer, cer_lo, cer_hi = bootstrap_ci(refs, hyps, "cer")
    wer, wer_lo, wer_hi = bootstrap_ci(refs, hyps, "wer")
    corpus = corpus_scores(refs, hyps)
    frame = pd.DataFrame(meta).assign(ref=refs, hyp=hyps)
    print(f"{tag}: CER {corpus['cer']:.3f} (utt mean {cer:.3f} "
          f"[{cer_lo:.3f}, {cer_hi:.3f}]) | WER {corpus['wer']:.3f} "
          f"[{wer_lo:.3f}, {wer_hi:.3f}] | n={len(refs)} | RTF {took / max(audio_s, 1e-6):.3f}")
    return frame


def slice_report(frame, min_n=20):
    buckets = {
        "all": frame,
        "code-switch": frame[frame["has_cs"]],
        "arabic only": frame[~frame["has_cs"]],
        "under 6s": frame[frame["duration"] < 6],
        "6 to 10s": frame[(frame["duration"] >= 6) & (frame["duration"] < 10)],
        "over 10s": frame[frame["duration"] >= 10],
    }
    rows = []
    for name, sub in buckets.items():
        if len(sub) < min_n:
            continue
        s = corpus_scores(sub["ref"], sub["hyp"])
        rows.append({"slice": name, "n": len(sub),
                     "cer": round(s["cer"], 3), "wer": round(s["wer"], 3)})
    return pd.DataFrame(rows)

print("eval helpers ready")


In [ ]:
RUN_EVAL = False
EVAL_CHECKPOINT = None      # None means evaluate the stock model
EVAL_GOLD_LIMIT = None      # e.g. 100 for a quick pass

if RUN_EVAL:
    if EVAL_CHECKPOINT:
        model, processor = load_student(EVAL_CHECKPOINT)
        label = f"student({Path(EVAL_CHECKPOINT).name})"
    else:
        model, processor = load_stock(BASE_MODEL, "cohere")
        label = "zero-shot"

    evaluate_split(model, processor, ds["validation"], f"{label} validation")
    evaluate_split(model, processor, ds["silver"], f"{label} silver")

    gold = load_gold("train", EVAL_GOLD_LIMIT)
    gold_frame = evaluate_split(model, processor, gold, f"{label} atlasia",
                                text_key="text")
    display(slice_report(gold_frame))
    gold_frame.to_csv(WORK_DIR / f"atlasia_{label.replace('/', '_')}.tsv",
                      sep="\t", index=False)
else:
    print("set RUN_EVAL = True (and EVAL_CHECKPOINT to a runs/.../best path)")


## 7. Where this stands

Two tables, because they answer different questions.

**Validation** (91 Gemini clips) is keep-best on `trainer_state.json`. Teacher agreement. Used to pick a checkpoint, not to report.

| recipe | encoder | decoder | CER | CER (norm) | WER | WER (norm) | trainable |
| --- | --- | --- | ---: | ---: | ---: | ---: | ---: |
| **hybrid** | MultiConv | LoRA | 15.6 | **14.9** | 34.4 | **31.8** | 21.1M |
| **full_lora** | LoRA | LoRA | 15.6 | **14.9** | 34.7 | **32.4** | 19.9M |
| encoder_lora | LoRA | frozen | 19.3 | 17.4 | 42.2 | 38.1 | 15.7M |
| decoder_lora | frozen | LoRA | 18.6 | 17.8 | 42.2 | 39.3 | 4.2M |

**AtlasIA** (114 human clips) is the number to report. Same `best/` checkpoints.

| recipe | encoder | decoder | CER | CER (norm) | WER | WER (norm) |
| --- | --- | --- | ---: | ---: | ---: | ---: |
| zero-shot | frozen | frozen | 22.7 | **20.2** | 53.9 | **49.1** |
| **hybrid** | MultiConv | LoRA | 15.3 | **14.4** | 42.2 | **38.3** |
| full_lora | LoRA | LoRA | 17.3 | 16.5 | 43.8 | 40.3 |
| encoder_lora | LoRA | frozen | 19.5 | 17.4 | 52.3 | 47.7 |
| decoder_lora | frozen | LoRA | 21.2 | 20.2 | 48.6 | 45.1 |

Open checkpoints (same seed and data):

| recipe | Hub |
| --- | --- |
| **hybrid** | [`01Yassine/cohere-transcribe-darija`](https://huggingface.co/01Yassine/cohere-transcribe-darija) |
| full_lora | [`01Yassine/cohere-transcribe-darija-full-lora`](https://huggingface.co/01Yassine/cohere-transcribe-darija-full-lora) |
| encoder_lora | [`01Yassine/cohere-transcribe-darija-encoder-lora`](https://huggingface.co/01Yassine/cohere-transcribe-darija-encoder-lora) |
| decoder_lora | [`01Yassine/cohere-transcribe-darija-decoder-lora`](https://huggingface.co/01Yassine/cohere-transcribe-darija-decoder-lora) |

On disk hybrid is `checkpoints/cohere-method/best`. `python infer.py clip.wav --model hybrid`

Reading this:

- You need both sides. Decoder-only (20.2) and encoder-only (17.4) barely move from the 20.2 zero-shot CER. The 14.4 only appears when MultiConv *and* decoder LoRA train together.
- **On AtlasIA the tie breaks.** Validation had hybrid and `full_lora` both at 14.9. On the human set hybrid is 14.4 against `full_lora` at 16.5. The conv adapter did something LoRA on the encoder did not.
- An earlier hybrid run with a smaller micro-batch is in `checkpoints/cohere-transcribe-arabic_lora_multiconv/`: validation CER 14.7 (norm 13.7). Same recipe, different batch, still Gemini.

What is still missing, in order:

1. Hand-correct 50 silver clips. Gemini's error rate against those corrections is the label noise floor. Gains smaller than that floor are not interpretable.
2. Read the slice table before the headline. Short clips and code-switch clips are where a Darija model usually breaks, and an average can improve while code-switch gets worse.
3. n=114 is small. Treat gaps of a couple of CER points as noise. One seed per recipe.

Known limits: labels are a teacher, not human annotation. Train audio is the clean end of YouTube while AtlasIA is field audio, so augmentation is doing real work. One seed per recipe. 91 validation clips have a bootstrap width of a few CER points, so treat 14.9 vs 14.9 as a tie, which is why gold eval exists.
